# Random Forest and Bagging

This notebook compares a Decision Tree, Random Forest, and Bagging Classifiers using the Titanic dataset. The original experiment is preserved; preprocessing leakage and reproducibility improvements will be addressed in a later revision.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
titanic = sns.load_dataset("titanic")

features = ["pclass", "sex", "fare", "embarked", "age"]
target = "survived"

X = titanic[features]
y = titanic[target]

# Split before fitting imputers/encoders to prevent preprocessing leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

numeric_features = ["pclass", "fare", "age"]
categorical_features = ["sex", "embarked"]

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

## Decision Tree baseline

A single Decision Tree provides a useful baseline. The preprocessing pipeline is fitted only on the training data.

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(max_depth=4, random_state=42))
])

model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

print("Training accuracy:", accuracy_score(y_train, y_pred_train))
print("Testing accuracy:", accuracy_score(y_test, y_pred_test))
print("\nClassification report:\n", classification_report(y_test, y_pred_test))

## Visualize the Decision Tree

In [ ]:
fitted_preprocessor = model.named_steps["preprocessor"]
fitted_tree = model.named_steps["classifier"]
feature_names = fitted_preprocessor.get_feature_names_out()

plt.figure(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=["Died", "Survived"],
    filled=True,
    rounded=True,
    max_depth=3
)
plt.title("Decision Tree (first three levels shown)")
plt.tight_layout()
plt.show()

## Random Forest and out-of-bag evaluation

Random Forest combines many trees. The OOB score estimates performance using observations left out of each tree's bootstrap sample; it complements, rather than replaces, held-out test evaluation.

In [ ]:
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=501,
        max_depth=4,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_estimator = rf.named_steps["classifier"]

print("OOB score:", rf_estimator.oob_score_)
print("Test accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))

## Bagging Classifier with a Decision Tree

In [ ]:
bagging_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=201,
        random_state=42,
        n_jobs=-1
    ))
])

bagging_tree.fit(X_train, y_train)
y_pred_bagging_tree = bagging_tree.predict(X_test)
print("Decision Tree Bagging accuracy:", accuracy_score(y_test, y_pred_bagging_tree))

## Bagging Classifier with Logistic Regression

In [ ]:
bagging_logistic = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000),
        n_estimators=201,
        random_state=42,
        n_jobs=-1
    ))
])

bagging_logistic.fit(X_train, y_train)
y_pred_bagging_logistic = bagging_logistic.predict(X_test)
print("Logistic Regression Bagging accuracy:", accuracy_score(y_test, y_pred_bagging_logistic))

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 10))
plot_tree(
    model,
    feature_names=X.columns,
    class_names=["Died", "Survived"],
    filled=True
)

plt.tight_layout()
plt.show()

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=501,
    oob_score=True,
    max_depth=4
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("OOB score: ", rf.oob_score_ * 100, "%")
print("testing accuracy: ", accuracy_score(y_test, y_pred) * 100, "%")

In [ ]:
# Bagging Classifier

from sklearn.ensemble import BaggingClassifier

base_model = DecisionTreeClassifier()

bagging = BaggingClassifier(
    base_model,
    n_estimators=201
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print("accuracy: ", accuracy_score(y_test, y_pred))

In [ ]:
# Bagging Classifier

from sklearn.linear_model import LogisticRegression

base_model = LogisticRegression(max_iter=1000)

bagging = BaggingClassifier(
    base_model,
    n_estimators=201
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print("accuracy: ", accuracy_score(y_test, y_pred))

## Compare model performance

In [ ]:
results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Bagging + Decision Tree", "Bagging + Logistic Regression"],
    "Test accuracy": [
        accuracy_score(y_test, model.predict(X_test)),
        accuracy_score(y_test, rf.predict(X_test)),
        accuracy_score(y_test, bagging_tree.predict(X_test)),
        accuracy_score(y_test, bagging_logistic.predict(X_test))
    ]
}).sort_values("Test accuracy", ascending=False)

results

## Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test)
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()

## Notes and next steps

- This notebook now splits the data before fitting imputation and encoding.
- `OneHotEncoder` avoids assigning artificial ordering to nominal categories such as sex and embarkation port.
- OOB score is available for the bootstrap-based Random Forest because `oob_score=True`.
- Further extensions: Bagging Regressor, Boosting, precision/recall comparisons, and feature importance analysis.